# Athenaeum (xenafiction.net): parsed sample for review

Notebook version of `fanfic-sample-athenaeum`. Parses search results pages 1 and 2 (100 records) so you can check the output before running a full crawl of about 140 pages.

**Network:** none if `verify_athenaeum.ipynb` (or `fanfic-verify-athenaeum`) has already saved both pages. Otherwise the missing pages are downloaded first (up to 2 requests, plus robots.txt).

In [ ]:
import os
import pathlib
import sys

# Find the repository root (the folder holding pyproject.toml) whether this
# notebook was opened from notebooks/ or from the root, and work from there,
# so saved pages and outputs always go to <repo>/data/ (or FANFIC_DATA_DIR).
REPO_ROOT = pathlib.Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("Open this notebook from inside the repository folder.")
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))  # works without `pip install -e .`
REPO_ROOT

## Settings

`CONTACT_EMAIL` is sent with every request (in the User-Agent) so site operators can reach whoever is crawling. Use your own address. Leave it blank to be asked the first time a download is needed, or set the `FANFIC_CONTACT_EMAIL` environment variable instead.

Only used if the search pages have not been saved yet.

In [ ]:
CONTACT_EMAIL = ""  # your email address

In [ ]:
import dataclasses
import json

from fanfic_scraper.discover.athenaeum import discover_athenaeum
from fanfic_scraper.fetch import ATHENAEUM, build_default_fetcher
from fanfic_scraper.parse.athenaeum import parse_athenaeum_search_page

fetcher = build_default_fetcher(ATHENAEUM, contact_email=CONTACT_EMAIL or None)
pages = discover_athenaeum(fetcher, max_pages=2)
records = [r for page in pages for r in parse_athenaeum_search_page(page.body, page.url)]

print(f"Parsed {len(records)} records from {len(pages)} Athenaeum search page(s)")

## Sample records

First 10 parsed records, full JSON.

In [ ]:
print(json.dumps([dataclasses.asdict(r) for r in records[:10]], indent=2, ensure_ascii=False))

## Search by title or author

Edit the query and re-run.

In [ ]:
def search(query, by="title"):
    q = query.lower()
    if by == "title":
        return [r for r in records if q in r.title.lower()]
    if by == "author":
        return [r for r in records if any(q in a.lower() for a in r.authors)]
    raise ValueError("by must be 'title' or 'author'")


for r in search("Quandary"):
    print(f"{r.title!r} by {r.authors} - status={r.status!r} ratings={r.ratings}")

## No-link stories (removed / published)

Records whose title has no story link: either "Removed By Author" or "Published" (sold commercially, not hosted here).

In [ ]:
no_link = [r for r in records if not r.parts and not r.external_url]
print(f"{len(no_link)} no-link record(s):")
for r in no_link:
    print(f"    {r.title!r} by {r.authors} - status={r.status!r} notes={r.notes!r}")

## External / off-site stories

Records whose title links out via `redir.cfm`. The real destination is stored; the redirect is never followed.

In [ ]:
external = [r for r in records if r.external_url]
print(f"{len(external)} external record(s):")
for r in external:
    print(f"    {r.title!r} by {r.authors} - external_url={r.external_url!r}")

## Ratings and ebook availability

`ratings` is `None` for stories below the site's minimum-votes threshold ("Fan Rating: N/A ... Needs N more"), never a partly filled dict.

In [ ]:
rated = [r for r in records if r.ratings]
ebooks = [r for r in records if r.ebook_available]
print(f"{len(rated)} of {len(records)} records have a visible fan rating")
print(f"{len(ebooks)} of {len(records)} records have ebook download links")
if rated:
    top = max(rated, key=lambda r: r.ratings["overall"])
    print(f"Highest-rated in this sample: {top.title!r} by {top.authors} - {top.ratings}")

## Write full output to disk

The same file `fanfic-sample-athenaeum` writes: all parsed records.

In [ ]:
from fanfic_scraper.fetch import default_data_dir

out_path = default_data_dir() / "athenaeum_sample_output.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump([dataclasses.asdict(r) for r in records], f, indent=2, ensure_ascii=False)
print(f"Wrote {len(records)} records to {out_path}")